<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Exercice_Gold_J1_W8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Part 1: Install llama-cpp-python with CUDA Support
To utilize the GPU in Google Colab, we need to set the `CMAKE_ARGS` to enable CUDA during the installation of `llama-cpp-python`.

In [ ]:
!CMAKE_ARGS="-DGGML_CUDA=ON" pip install llama-cpp-python --verbose
!pip install huggingface_hub

### Part 2: Download Llama 3.1 8B (GGUF Format)
We will download the Llama 3.1 8B Instruct model in GGUF format from Hugging Face.

In [ ]:
from huggingface_hub import hf_hub_download

# Downloading Llama 3.1 8B Instruct (Q4_K_M quantization for balance of speed and quality)
model_path = hf_hub_download(
    repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
    filename="Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
)

print(f"Model downloaded to: {model_path}")

### Part 3: Inference with Llama 3.1
Now we initialize the model and perform both standard and streaming generation.

In [ ]:
from llama_cpp import Llama

# Initialize the model
# n_gpu_layers=-1 moves all layers to the GPU
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,
    n_ctx=2048,
    verbose=False
)

# 1. Simple text generation
prompt = "Q: Explain how the solar system formed in two sentences? A:"
output = llm(prompt=prompt, max_tokens=100)
print("--- Standard Output ---")
print(output["choices"][0]["text"])

# 2. Streaming code generation
print("\n--- Streaming Code Generation ---")
stream_prompt = "Write a Python function to check if a number is prime."
output_stream = llm(
    prompt=f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{stream_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
    max_tokens=256,
    stream=True
)

for chunk in output_stream:
    token = chunk["choices"][0]["text"]
    print(token, end="", flush=True)